# Weather Prediction and Advisory System
Forecasting rainfall using Logistic Regression, Decision Tree, and XGBoost with interactive dashboards.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.feature_selection import SelectKBest, chi2
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Explore Data

In [ ]:
df = pd.read_csv('weatherAUS.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0])

## 3. Data Cleaning

In [ ]:
# Drop rows where target is missing
df.dropna(subset=['RainTomorrow'], inplace=True)

# Drop Date and Location columns
df.drop(columns=['Date', 'Location'], inplace=True)

# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print('Numerical columns:', num_cols)
print('Categorical columns:', cat_cols)

In [ ]:
# Fill missing values
# Numerical - fill with median
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical - fill with mode
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print('Missing values after cleaning:')
print(df.isnull().sum().sum())

In [ ]:
# Encode categorical columns
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

df.head()

## 4. Exploratory Data Analysis

In [ ]:
# Rain distribution
fig = px.pie(df, names='RainTomorrow', title='Rain Tomorrow Distribution',
             color_discrete_sequence=['#636EFA', '#EF553B'])
fig.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), cmap='coolwarm', annot=False)
plt.title('Feature Correlation Heatmap')
plt.show()

In [ ]:
# Humidity vs Rain
fig = px.box(df, x='RainTomorrow', y='Humidity3pm',
             title='Humidity at 3pm vs Rain Tomorrow',
             color='RainTomorrow')
fig.show()

In [ ]:
# Temperature vs Rain
fig = px.box(df, x='RainTomorrow', y='MaxTemp',
             title='Max Temperature vs Rain Tomorrow',
             color='RainTomorrow')
fig.show()

## 5. Feature Selection

In [ ]:
X = df.drop('RainTomorrow', axis=1)
y = df['RainTomorrow']

# SelectKBest - select top 15 features
selector = SelectKBest(score_func=chi2, k=15)

# chi2 needs non-negative values, use abs
X_abs = X.abs()
selector.fit(X_abs, y)

feature_scores = pd.DataFrame({'Feature': X.columns, 'Score': selector.scores_})
feature_scores = feature_scores.sort_values('Score', ascending=False).head(15)

fig = px.bar(feature_scores, x='Score', y='Feature', orientation='h',
             title='Top 15 Features by Importance')
fig.show()

# Select top features
selected_features = feature_scores['Feature'].tolist()
X = df[selected_features]
print('Selected features:', selected_features)

## 6. Train-Test Split and Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train size:', X_train.shape)
print('Test size:', X_test.shape)

## 7. Model Building and Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred)
    
    results.append({'Model': name, 'Accuracy': round(acc, 4), 'AUC Score': round(auc, 4)})
    print(f'\n===== {name} =====')
    print(f'Accuracy: {acc:.4f}')
    print(f'AUC Score: {auc:.4f}')
    print(classification_report(y_test, y_pred))

results_df = pd.DataFrame(results)
print('\n', results_df)

## 8. Model Comparison Dashboard

In [ ]:
# Accuracy comparison
fig = px.bar(results_df, x='Model', y='Accuracy',
             title='Model Accuracy Comparison',
             color='Model', text='Accuracy')
fig.show()

In [ ]:
# Confusion matrix for best model (XGBoost)
xgb_model = models['XGBoost']
y_pred_xgb = xgb_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_xgb)

fig = px.imshow(cm, text_auto=True,
                labels=dict(x='Predicted', y='Actual'),
                x=['No Rain', 'Rain'], y=['No Rain', 'Rain'],
                title='XGBoost Confusion Matrix')
fig.show()

## 9. Real-time Prediction

In [ ]:
def predict_rain(input_values):
    """
    Predict if it will rain tomorrow.
    input_values: dict with feature values
    """
    input_df = pd.DataFrame([input_values])
    input_scaled = scaler.transform(input_df)
    prediction = xgb_model.predict(input_scaled)[0]
    probability = xgb_model.predict_proba(input_scaled)[0][1]
    
    result = 'RAIN' if prediction == 1 else 'NO RAIN'
    print(f'Prediction: {result}')
    print(f'Rain Probability: {round(probability * 100, 2)}%')
    return result

# Example prediction - use median values from dataset
sample_input = {col: float(df[col].median()) for col in selected_features}
predict_rain(sample_input)